In [1]:
import sys
BASE_DIR = "../../../.."
sys.path.insert(0, BASE_DIR)

import pandas as pd
import numpy as np
import ast
import random
import json
from time import time
import gc
import chromadb
from tqdm import tqdm
from dataclasses import dataclass, field
from sentence_transformers import SentenceTransformer
from typing import Dict, List
from dataclasses import dataclass

random.seed(42)

from src.agents.hosted import AgentConnector, AgentConnectorConfig, AgentConnectionType, LocalAgentConnectionParams, AgentModelConfig
from src.utils import ReaderMetrics

VECTOR_DB_PATH = '../../../../data/natural_questions/dbs/v3/densedb'
CHUNKS_PATH = "../../../../data/natural_questions/natural_qa_chunked/chunked_nqa1.csv"
BASE_DATASET_PATH = "../../../../data/natural_questions/natural_q1.csv"
AGENT_MODEL_PATH = "../../../../models/Undi95/Meta-Llama-3-8B-Instruct-hf"

/home/jovyan/work/alexander_workspace/conda/envs/rag_alex/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [49]:
PARAMS = {
    'version': 3,
    'num_samples': 2000,
    'num_contexts': 1,
    'system_prompt': "You are an AI assistant who helps solve user issues.",
    "item_format": "- {document}",
    "user_prompt": 'Answer the question using the available information from the texts in the list below. Generate answer only in English. Do not duplicate the question in the answer. Generate only the answer to the specified question. Answer need to be short. Do not generate anything extra.',
    "prompt_format": "{user_p}\n\nAvailable information:\n{cnt_list}\n\nQuestion:\n{q}\n\nAnswer:\n",
    'scores': {'rel': 1.0, 'unrel': 0.0},
    'gen_strat': {'max_new_tokens': 1024},
    'stub_answer': "I do not have an answer to your question"
}

DB_NAME = 'natural_questions'

METADATA_SAVE_NAME = 'metadata.json'
USER_PROPMTS_SAVE_NAME = 'user_prompts.json'
PARAMS_SAVE_NAME = 'hyperp.json'
GEN_ANSW_SAVE_NAME = 'generation_info.json'
SCORES_SAVE_NAME = 'scores.json'

### Подключение к векторной бд

In [3]:
@dataclass
class EmbedderModelConfig:
    model_name_or_path: str = '../../../../models/multilingual-e5-small'
    prompts: Dict = field(default_factory=lambda: {"query": "query: ", "passage": "passage: "})
    device: str = 'cuda'
    normalize_embeddings: bool = True

class EmbedderModel:
    def __init__(self, config: EmbedderModelConfig = EmbedderModelConfig()) -> None:
        self.config = EmbedderModelConfig() if config is None else config
        self.model = SentenceTransformer(
            config.model_name_or_path, device=config.device,
            prompts=config.prompts
        )

    def encode_queries(self, queries: List[str], **kwargs) -> List[List[float]]:
        output = self.model.encode(queries, prompt_name='query', 
                                 normalize_embeddings=self.config.normalize_embeddings, **kwargs)
        return output

    def encode_passages(self, passages: List[str], **kwargs) -> List[List[float]]:
        output = self.model.encode(passages, prompt_name='query',
                                 normalize_embeddings=self.config.normalize_embeddings,
                                 **kwargs)
        return [list(obj.astype(float)) for obj in output]

In [4]:
client = chromadb.PersistentClient(path=VECTOR_DB_PATH)
collection = client.get_collection(name=DB_NAME)
embedder = EmbedderModel()
print(collection.count())

164864


### Подключение к агенту

In [5]:
agent_config = AgentConnectorConfig(
    agent_config = AgentModelConfig(model_name_or_path = "../../../../models/Undi95/Meta-Llama-3-8B-Instruct-hf"))

In [6]:
agent = AgentConnector.open(agent_config)

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

In [7]:
print(agent.generate("what is wrong with humanity?"))

Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


What a profound and complex question! As a helpful assistant, I'll provide some insights and perspectives, but please note that there's no one-size-fits-all answer. Humanity is a diverse and multifaceted species, and what might be a problem for one person or group might not be the same for another.

That being said, here are some common issues that have been identified as challenges for humanity:

1. **Conflict and violence**: Wars, terrorism, and other forms of violence have plagued human history, causing immense suffering and destruction.
2. **Inequality and social injustice**: Systemic inequalities based on factors like race, gender, class, and religion can lead to discrimination, marginalization, and social unrest.
3. **Environmental degradation**: Human activities have significantly harmed the planet, leading to climate change, pollution, and loss of biodiversity.
4. **Mental health and well-being**: Many people struggle with mental health issues, such as depression, anxiety, and 

### Формируем список контекстов для каждого запроса со скорами

In [8]:
dataset_df = pd.read_csv(BASE_DATASET_PATH)

In [11]:
CONTEXTS_LIST_IDS = []
for i in tqdm(range(PARAMS['num_samples'])):
    
    # retrieving relevant chunk
    emb_query = embedder.encode_queries([dataset_df['question'][i]])[0]
    output = collection.query(
        query_embeddings=[emb_query.tolist()],
        include=['metadatas'], n_results=1)

    cur_list_ids = [(-1, output['metadatas'][0][0]['chunk_index'])]

    CONTEXTS_LIST_IDS.append(cur_list_ids)

100%|██████████| 2000/2000 [37:07<00:00,  1.11s/it]


### Готовим промпт

In [50]:
chunks_df = pd.read_csv(CHUNKS_PATH)

In [51]:
USER_PROMPTS = []
gc.collect()
for i in tqdm(range(len(CONTEXTS_LIST_IDS))):
    rel_doc = chunks_df['chunk'][CONTEXTS_LIST_IDS[i][0][1]]
    documents_list = PARAMS['item_format'].format(document=rel_doc)
    
    USER_PROMPTS.append(PARAMS['prompt_format'].format(user_p=PARAMS['user_prompt'], cnt_list=documents_list, q=dataset_df['question'][i]))

100%|██████████| 2000/2000 [00:00<00:00, 84602.66it/s]


In [53]:
with open(f"./logs/v{PARAMS['version']}/{USER_PROPMTS_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps(USER_PROMPTS, ensure_ascii=False, indent=1))

# сохраняем конфигурацию эксперимента
with open(f"./logs/v{PARAMS['version']}/{PARAMS_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps(PARAMS, ensure_ascii=False, indent=1))

In [54]:
print(USER_PROMPTS[0])

Answer the question using the available information from the texts in the list below. Generate answer only in English. Do not duplicate the question in the answer. Generate only the answer to the specified question. Answer need to be short. Do not generate anything extra.

Available information:
- The Lost and the Plunderers '' March 4 , 2018 TBD TBD TBD TBD TBD TBD 11 `` Dead or Alive Or '' March 11 , 2018 TBD TBD TBD TBD TBD TBD 12 `` The Key '' March 18 , 2018 TBD TBD TBD TBD TBD TBD ^ 1 Live + 7 ratings were not available , so Live + 3 ratings have been used instead . References ( edit ) Jump up ^ Schwartz , Ryan ( July 18 , 2017 ) . `` Walking Dead Gets Season 8 Premiere Date , Plus Fear TWD Return Date '' . TVLine . Retrieved July 18 , 2017 . ^ Jump up to : Hibberd , James (

Question:
when is the last episode of season 8 of the walking dead

Answer:



In [55]:
del chunks_df
gc.collect()

920

### Генерируем ответы на вопросы

In [56]:
generate_answers = []
display_iter = 100
s_time = time()
for i in tqdm(range(len(USER_PROMPTS))):
    pred_answer = agent.generate(user_prompt=USER_PROMPTS[i], system_prompt=PARAMS['system_prompt'], gen_strategy=PARAMS['gen_strat'])
    generate_answers.append(pred_answer)

    if i % display_iter == 0:
        print(f"\n[{i}]: \nGEN: {pred_answer}\nGOLD: {dataset_df['short_answer'][i]}")
e_time = time()

  0%|          | 1/2000 [00:00<22:31,  1.48it/s]


[0]: 
GEN: March 18, 2018
GOLD: March 18, 2018


  5%|▌         | 101/2000 [01:49<36:39,  1.16s/it] 


[100]: 
GEN: There is no information about the Ford Fusion in the provided texts. The texts only discuss the movie "Rise of the Guardians" and its related topics.
GOLD: gasoline-electric


 10%|█         | 201/2000 [04:13<46:18,  1.54s/it]  

...

 90%|█████████ | 1801/2000 [41:26<05:14,  1.58s/it]


[1800]: 
GEN: There is no information provided about "America's Got Talent" or its host. The text only discusses the geography and regions of Thailand.
GOLD: Tyra Banks


 95%|█████████▌| 1901/2000 [43:55<01:20,  1.22it/s]


[1900]: 
GEN: There is no mention of the vena cava in the provided text.
GOLD: large vein that carries deoxygenated blood from the lower and middle body into the right atrium of the heart.


100%|██████████| 2000/2000 [46:22<00:00,  1.39s/it]


In [57]:
# сохраняем используемые контексты + сгнерированные ответы
gen_info = []
for i in range(PARAMS['num_samples']):
    cur_item = {'gen_answer': generate_answers[i], 'used_contexts': CONTEXTS_LIST_IDS[i]}
    gen_info.append(cur_item)

with open(f"./logs/v{PARAMS['version']}/{GEN_ANSW_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps(gen_info, ensure_ascii=False, indent=1))

with open(f"./logs/v{PARAMS['version']}/{METADATA_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps({'elapsed_time': e_time - s_time}, ensure_ascii=False, indent=1))

### Оцениваем качество

In [58]:
import nltk
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [59]:
LOADING_VERSION = "3"

In [60]:
with open(f'./logs/v{LOADING_VERSION}/{GEN_ANSW_SAVE_NAME}','r', encoding='utf8') as fd:
    predicted_answers = list(map(lambda v: v['gen_answer'], json.loads(fd.read())))

In [42]:
metrics = ReaderMetrics(base_dir=BASE_DIR, model_path='en_electra_base')

Loading Meteor...
Loading ExactMatch


In [43]:
dataset_df = pd.read_csv(BASE_DATASET_PATH)

In [61]:
target_scores = {
    'BLEU2': [], 'BLEU1': [],
    'ExactMatch': [],'METEOR': [],
    'BertScore': [],
    'Levenshtain': [],
    'ROUGEL': []}

stub_scores = {
    'BLEU2': [], 'BLEU1': [],
    'ExactMatch': [],'METEOR': [],
    'BertScore': [],
    'Levenshtain': [],
    'ROUGEL': []}

show_step = 10

process = tqdm(range(PARAMS['num_samples']))
target_answers =  dataset_df['short_answer'].to_list()[:PARAMS['num_samples']]
tmp_stub_pred_answers = []
for i in process:
    
    predicted_answer = predicted_answers[i]
    target_answer = target_answers[i]

    target_scores['BLEU1'] += metrics.bleu1([predicted_answer], [target_answer])
    target_scores['BLEU2'] += metrics.bleu2([predicted_answer], [target_answer])
    target_scores['ExactMatch'] += metrics.exact_match([predicted_answer], [target_answer])
    target_scores['METEOR'] += metrics.meteor([predicted_answer], [target_answer])
    target_scores['Levenshtain'] += metrics.levenshtain_score([predicted_answer], [target_answer])
    target_scores['ROUGEL'] += metrics.rougel([predicted_answer], [target_answer])


    stub_pred_answer = predicted_answer
    tmp_stub_pred_answers.append(stub_pred_answer)

    stub_scores['BLEU1'] += metrics.bleu1([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['BLEU2'] += metrics.bleu2([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['ExactMatch'] += metrics.exact_match([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['METEOR'] += metrics.meteor([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['Levenshtain'] += metrics.levenshtain_score([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['ROUGEL'] += metrics.rougel([stub_pred_answer], [PARAMS['stub_answer']])
            
    if i % show_step == 0:
        process.set_postfix({m_name: np.mean(score) for m_name, score in stub_scores.items()})

target_scores = {m_name: round(float(np.mean(score)), 5) for m_name, score in target_scores.items()}
target_scores['BertScore'] = metrics.bertscore(predicted_answers, target_answers)

stub_scores = {m_name: round(float(np.mean(score)), 5) for m_name, score in stub_scores.items()}
stub_scores['BertScore'] = metrics.bertscore(tmp_stub_pred_answers, [PARAMS['stub_answer']]*len(tmp_stub_pred_answers))
stub_scores['elapsed_time_sec'] = round(float(process.format_dict["elapsed"]), 3)

  0%|          | 0/2000 [00:00<?, ?it/s]/home/jovyan/work/alexander_workspace/conda/envs/rag_alex/lib/python3.10/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/jovyan/work/alexander_workspace/conda/envs/rag_alex/lib/python3.10/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
100%|██████████| 2000/2000 [25:29<00:00,  1.31it/s, BLEU2=0.00639, BLEU1=0.041, ExactMatch=0, METEOR=0.069, BertScore=nan, Levenshtain=114, ROUGEL=0.052]   


In [62]:
with open(f"./logs/v{PARAMS['version']}/{SCORES_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps({'target_answers': target_scores, 'stub_answers': stub_scores}, ensure_ascii=False, indent=1))